## Import Headers

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json
from dotenv import load_dotenv
load_dotenv()

from helpers import extract_ordered_content

## Compliance Reasoning Prompt Template (Regulations vs Vendor Report)

In [8]:
llm = ChatOpenAI(model="gpt-4o")

template = """
You are a compliance analyst.

Return the answer strictly in the following format:

Overall: <Compliant / Non-Compliant>

Test-wise Breakdown:
For each test condition found in the documents:
- <Test Name>: <Compliant / Non-Compliant>. 

Explanation:
- Mention only the components that violate their limits (if any).
- Show the numeric comparison (measured vs threshold).
- Calculate the percentage of non-compliant components for that test.
- Apply the batch rejection rule exactly as stated in the regulations.
- Keep the explanation concise and structured.

Important:
- Do NOT repeat full regulatory tables.
- Do NOT repeat all measurements unless necessary.
- Focus only on violations and final reasoning.

---

Regulatory thresholds:
{us_regulations}

Vendor measurements:
{vendor_measurements}

Question:
{question}
"""

prompt = PromptTemplate(
    input_variables=["us_regulations", "vendor_measurements", "question"],
    template=template
)

## Extract contents of PDF using PymuPDF (fitz) and pdfplumber:
- If Text  → keep as-is
- If Table → convert to markdown
- If Image → send to llm and get caption.

In [11]:
def run_compliance_pipeline(vendor_pdf_path, regulation_pdf_path, question):
    
    print("🔹 Step 1: Extracting Vendor Document...\n")
    vendor_content = extract_ordered_content(vendor_pdf_path)
    print("Vendor extraction complete.")
    print(json.dumps(vendor_content, indent=2))
    
    print("\n🔹 Step 2: Extracting Regulation Document...\n")
    regulation_content = extract_ordered_content(regulation_pdf_path)
    print("Regulation extraction complete.")
    print(json.dumps(regulation_content, indent=2))
    
    print("\n🔹 Step 3: Building Prompt...\n")
    final_prompt = prompt.format(
        us_regulations=regulation_content,
        vendor_measurements=vendor_content,
        question=question
    )
    print("Prompt built successfully.")
    
    print("\n🔹 Step 4: Running LLM Compliance Reasoning...\n")
    response = llm.invoke(final_prompt)
    
    print("\n🔹 Step 5: Final Compliance Decision\n")
    print(response.content)
    
    return response.content


## Run Compliance Pipeline

> Note: Exactly one component measurement was intentionally set above the threshold in the mud test to validate the robustness of the pipeline.

In [12]:
run_compliance_pipeline(
    vendor_pdf_path="data-reports/vendor_report.pdf",
    regulation_pdf_path="data-reports/us_regulations.pdf",
    question="Is the batch compliant under all tests?"
)

🔹 Step 1: Extracting Vendor Document...

Vendor extraction complete.
[
  {
    "type": "text",
    "content": "Water Test",
    "page": 1,
    "y": 76.73999786376953
  },
  {
    "type": "text",
    "content": "All tested components demonstrated vibration levels within acceptable US regulatory limits for water test.",
    "page": 1,
    "y": 126.8499755859375
  },
  {
    "type": "image",
    "content": "[IMAGE DESCRIPTION]: Bar chart titled \"4.1 Water Test Vibration Levels\" displaying vibration levels (mm/s) for five part codes: L102 (2.23), M330 (3.77), X778 (2.57), A450 (2.37), B990 (2.99).",
    "page": 1,
    "y": 187.5999755859375
  },
  {
    "type": "text",
    "content": "Dust Test",
    "page": 2,
    "y": 76.73999786376953
  },
  {
    "type": "text",
    "content": "All tested components demonstrated vibration levels within acceptable US regulatory limits for dust test.",
    "page": 2,
    "y": 126.8499755859375
  },
  {
    "type": "image",
    "content": "[IMAGE DESCRI

'Overall: Non-Compliant\n\nTest-wise Breakdown:\n- Water Test: Compliant.\n- Dust Test: Compliant.\n- Mud Test: Non-Compliant.\n\nExplanation:\n- Mud Test:\n  - X778: Measured 4.8 mm/s vs threshold 4.0 mm/s.\n  - Violation percentage: 20% (1 out of 5 components).\n- According to the Batch-Level Rejection Rule, the entire batch is rejected because more than 10% of tested components exceeded their thresholds in the Mud Test.'